# ARC Colab Model Host

Turn this Colab runtime into ARC's **model host** — it runs Ollama, pulls the full council, and serves inference to your ARC app whenever it's online. **No billing needed** (free T4 GPU).

## Setup
1. Runtime → Change runtime type → **T4 GPU** (free).
2. (Optional) Paste your ARC API key into `ARC_API_KEY` below. ARC runs open by default, so you can also leave it empty.
3. Runtime → **Run all**. First run downloads the models (~10–15 min).
4. Keep this tab open. ARC auto-detects the worker within seconds and switches the council to:
   - **qwen2.5:7b-instruct** — reasoning & coding
   - **dolphin-llama3:8b** — creative / unrestricted takes
   - **dolphin-mistral:7b-v2.8-q3_K_M** — fast perspective & cross-checking

When the notebook stops, ARC falls back to its always-on Render host automatically — no errors, just a smaller brain.

In [ ]:
import json, time, uuid, sys, io, traceback, subprocess, requests

# ===== ARC Colab Model Host Worker =====
ARC_URL = "https://arc-api-d151.onrender.com"
ARC_API_KEY = ""   # <-- paste your ARC API key here

# The mixed council — each model contributes a different strength
MODELS = [
    "qwen2.5:7b-instruct",             # reasoning + coding
    "dolphin-llama3:8b",               # creative / unrestricted
    "dolphin-mistral:7b-v2.8-q3_K_M",  # fast perspective
]

H = {"Authorization": f"Bearer {ARC_API_KEY}"} if ARC_API_KEY else {}
BASE = f"{ARC_URL}/colab"

GPU = ""
try:
    import torch
    GPU = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    GPU = "cpu"

# --- 1) install + start ollama in this VM ---
print("Installing Ollama...")
subprocess.run(["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"], check=False)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def ollama_ready():
    try:
        return requests.get("http://127.0.0.1:11434/api/version", timeout=5).ok
    except Exception:
        return False

for _ in range(60):
    if ollama_ready():
        break
    time.sleep(2)
print("Ollama server up.")

# --- 2) pull the council models ---
for m in MODELS:
    print(f"Pulling {m} ...")
    subprocess.run(["ollama", "pull", m])
print("Council models ready.")

# --- 3) register as an ARC worker serving these models ---
def register():
    r = requests.post(BASE, json={"action": "register",
                                   "name": f"colab-{uuid.uuid4().hex[:4]}",
                                   "gpu": GPU, "models": MODELS},
                       headers=H, timeout=30)
    r.raise_for_status()
    return r.json()["id"]

WORKER = register()
print(f"ARC worker ONLINE: {WORKER} (gpu={GPU}) — serving {MODELS}")

# --- 4) work loop: heartbeat with model list + claim jobs ---
while True:
    try:
        hb = requests.post(BASE, json={"action": "heartbeat", "worker_id": WORKER,
                                       "models": MODELS}, headers=H, timeout=30)
        if hb.status_code == 404:
            WORKER = register()   # ARC restarted and forgot us
            print(f"re-registered as {WORKER}")
        job = requests.post(BASE, json={"action": "next", "worker_id": WORKER},
                            headers=H, timeout=30).json().get("job")
        if job:
            print(f">> claimed job {job['id']}: {job['title']}")
            ok, out = True, ""
            buf, old = io.StringIO(), sys.stdout
            try:
                sys.stdout = buf
                exec(compile(job["code"], f"arc-job-{job['id']}", "exec"), {})
            except Exception:
                ok = False
                out = traceback.format_exc()
            finally:
                sys.stdout = old
                out = (buf.getvalue() + "\n" + out).strip()[:20000]
            requests.post(BASE, json={"action": "result", "worker_id": WORKER,
                                     "job_id": job["id"], "ok": ok, "output": out},
                          headers=H, timeout=60)
            print(("OK\n" if ok else "FAILED\n") + out[:400] + "\n" + "-" * 60)
        time.sleep(3)
    except KeyboardInterrupt:
        print("worker stopped")
        break
    except Exception as e:
        print("loop error:", type(e).__name__)
        time.sleep(10)